# KOSIS Chroma Coordinate Mapping Colab

이 노트북은 KOSIS 2차 READY 단계에서 Chroma 좌표검색을 적용해 기존 lexical 결과와 비교하기 위한 Colab 실행용입니다.

핵심 원칙:
- Chroma/BGE/reranker는 ITEM/OBJ 후보 생성과 순위화에만 사용합니다.
- 최종 READY 확정은 기존 공식 KOSIS 메타 + KOSIS API exact validation을 그대로 사용합니다.
- margin이 작은 후보는 READY가 아니라 PROVISIONAL로 분리합니다.


## 1. Google Drive 마운트

아래 경로는 예시입니다. Drive에 프로젝트 폴더를 올린 뒤 `PROJECT_DIR`만 맞춰주세요.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# TODO: 본인 Drive 경로에 맞게 수정
PROJECT_DIR = '/content/drive/MyDrive/NLP_05-Team-Project-3'
%cd $PROJECT_DIR


## 2. 의존성 설치

Colab에서는 chromadb, sentence-transformers, torch 등이 필요합니다.


In [ ]:
!pip install -q -r requirements-ml.txt


## 3. 입력 파일 경로 설정

아래 2개 파일은 기존 파이프라인에서 이미 만들어진 파일입니다.

- `*_kosis_meta_index.csv`
- `*_kosis_candidates_with_meta.csv`

기존 lexical 검증 결과가 있으면 `BASE_VALIDATED`와 `BASE_VERIFIED`도 같이 비교합니다.


In [ ]:
from pathlib import Path

# kosis_mapping_ready_500_real.csv 기준 실행
# GitHub DBME 브랜치를 clone하면 샘플 입력 파일은 repo 안에 있다고 가정합니다.
# Drive에 따로 올렸다면 아래 INPUT_CSV만 /content/drive/MyDrive/... 경로로 바꾸세요.
INPUT_CSV = Path('kosis_mapping_ready_500_real.csv')
RUN_DIR = Path('kosis_mapping_ready_500_real_run')
RUN_DIR.mkdir(parents=True, exist_ok=True)

BASE_NAME = INPUT_CSV.stem
META_INDEX = RUN_DIR / f'{BASE_NAME}_kosis_meta_index.csv'
CANDIDATES_WITH_META = RUN_DIR / f'{BASE_NAME}_kosis_candidates_with_meta.csv'
BASE_VALIDATED = RUN_DIR / f'{BASE_NAME}_kosis_validated_mappings.csv'
BASE_VERIFIED = RUN_DIR / f'{BASE_NAME}_kosis_verified.csv'

CHROMA_DIR = Path('data/indexes/kosis_chroma_coordinates_colab')
CHROMA_VALIDATED = RUN_DIR / f'{BASE_NAME}_kosis_validated_mappings_chroma.csv'
CHROMA_VERIFIED = RUN_DIR / f'{BASE_NAME}_kosis_verified_chroma.csv'

print('INPUT_CSV:', INPUT_CSV, 'exists=', INPUT_CSV.exists())
print('RUN_DIR:', RUN_DIR)


## 4. 기본 파이프라인 먼저 실행

`kosis_mapping_ready_500_real.csv`는 1차 IN_READY 입력 파일입니다. 먼저 기존 2차 매핑/검증 산출물(`meta_index`, `candidates_with_meta`, `validated`, `verified`)을 만듭니다.


In [ ]:
!python run_kosis_measurement_pipeline.py \
  --input "$INPUT_CSV" \
  --table-index kosis_table_summary.csv \
  --out-dir "$RUN_DIR" \
  --retrieval-mode lexical \
  --verify \
  --item-top-k 3 \
  --obj-top-k 2 \
  --max-combinations 20 \
  --delay 0.05


## 5. Chroma 좌표 인덱스 생성

기본 파이프라인에서 만들어진 `*_kosis_meta_index.csv`를 좌표 document로 변환해 Chroma 인덱스를 만듭니다.

처음 전체 인덱스가 너무 오래 걸리면 `--max-documents 50000`을 임시로 추가해서 빠른 비교부터 하세요.


In [ ]:
!python kosis_build_chroma_coordinate_index.py \
  --meta-index "$META_INDEX" \
  --persist-dir "$CHROMA_DIR" \
  --batch-size 256 \
  --force


In [ ]:
!python kosis_validate_mapping_candidates.py \
  --input "$CANDIDATES_WITH_META" \
  --meta-index "$META_INDEX" \
  --output "$CHROMA_VALIDATED" \
  --chroma-coordinate-index "$CHROMA_DIR" \
  --item-top-k 3 \
  --obj-top-k 2 \
  --max-combinations 20 \
  --delay 0.05


## 7. 실제값 검증: VERIFIED 생성

Chroma 적용 후 READY로 통과한 후보를 실제 KOSIS 값과 비교합니다.


In [ ]:
!python kosis_verify_claim_values.py \
  --input "$CHROMA_VALIDATED" \
  --output "$CHROMA_VERIFIED" \
  --delay 0.05


## 8. 기존 결과 vs Chroma 결과 비교

READY / PROVISIONAL / VERIFIED 수를 비교합니다.


In [ ]:
import csv
from collections import Counter
from pathlib import Path

def read_rows(path):
    path = Path(path)
    if not path.exists():
        print('missing:', path)
        return []
    with path.open(encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))

def summarize_validated(label, path):
    rows = read_rows(path)
    print('\n##', label)
    print('rows:', len(rows))
    print('mapping_status:', Counter(r.get('mapping_status','') for r in rows))
    print('mapping_reason top10:', Counter(r.get('mapping_reason','') for r in rows).most_common(10))
    return rows

def summarize_verified(label, path):
    rows = read_rows(path)
    print('\n##', label)
    print('rows:', len(rows))
    print('verdict:', Counter(r.get('verdict','') for r in rows))
    print('verdict_code:', Counter(r.get('verdict_code','') for r in rows))
    return rows

base_validated_rows = summarize_validated('BASE VALIDATED', BASE_VALIDATED)
chroma_validated_rows = summarize_validated('CHROMA VALIDATED', CHROMA_VALIDATED)
base_verified_rows = summarize_verified('BASE VERIFIED', BASE_VERIFIED)
chroma_verified_rows = summarize_verified('CHROMA VERIFIED', CHROMA_VERIFIED)


## 9. READY로 새로 올라간 행 확인

기존에는 READY가 아니었는데 Chroma 적용 후 READY/PROVISIONAL이 된 measurement를 확인합니다.


In [ ]:
def row_key(r):
    return r.get('claim_measurement_id') or r.get('claim_id') or ''

base_by_key = {row_key(r): r for r in base_validated_rows}
interesting = []
for r in chroma_validated_rows:
    k = row_key(r)
    before = base_by_key.get(k, {})
    before_status = before.get('mapping_status','')
    after_status = r.get('mapping_status','')
    if before_status != after_status and after_status in {'READY', 'PROVISIONAL', 'NEEDS_CONFIRMATION'}:
        interesting.append((k, before_status, after_status, r.get('tbl_id',''), r.get('selected_itm_id',''), r.get('selected_obj_l1',''), r.get('mapping_reason','')))

print('changed interesting rows:', len(interesting))
for row in interesting[:50]:
    print(row)
